Import libraries

In [2]:
import os
import time
import random
import json
import copy
import shutil
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 80)
print("NOTEBOOK 8 — IMAGE-CONDITIONED CNN POLICY MODEL")
print("=" * 80)

print()
print("✓ Required libraries imported.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("GPU             : Not available")

NOTEBOOK 8 — IMAGE-CONDITIONED CNN POLICY MODEL

✓ Required libraries imported.
PyTorch version : 2.12.0.dev20260226+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5060 Ti


Configuration next.

In [3]:
BASE_DIR = Path(r"E:\ML_Project")

TRAIN_IMAGE_DIR = (
    BASE_DIR /
    "datasets" /
    "bdd100k_original_yolo" /
    "train" /
    "images"
)
ORACLE_FILE = (
    BASE_DIR /
    "results" /
    "oracle_labels" /
    "policy_train_00000_70000.csv"
)
OUTPUT_DIR = (
    BASE_DIR /
    "results" /
    "final_analysis" /
    "cnn_policy"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)
ACTIONS = [
    "Identity",
    "CLAHE",
    "Gamma",
    "Dehazing",
    "Denoising"
]
ACTION_TO_ID = {
    "Identity": 0,
    "CLAHE": 1,
    "Gamma": 2,
    "Dehazing": 3,
    "Denoising": 4
}
ID_TO_ACTION = {
    0: "Identity",
    1: "CLAHE",
    2: "Gamma",
    3: "Dehazing",
    4: "Denoising"
}

RANDOM_STATE = 42
TRAIN_RATIO = 0.80
IMAGE_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 4

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("NOTEBOOK 8 — CONFIGURATION")
print("=" * 80)

print()
print(f"Project directory : {BASE_DIR}")
print(f"Training images   : {TRAIN_IMAGE_DIR}")
print(f"Oracle dataset    : {ORACLE_FILE}")
print(f"Output directory  : {OUTPUT_DIR}")
print(f"Model directory   : {MODEL_DIR}")

print()
print("Actions:")
for action_id, action_name in ID_TO_ACTION.items():
    print(f"{action_id} : {action_name}")

print()
print(f"Image size        : {IMAGE_SIZE} × {IMAGE_SIZE}")
print(f"Batch size        : {BATCH_SIZE}")
print(f"Workers           : {NUM_WORKERS}")
print(f"Train ratio       : {TRAIN_RATIO}")
print(f"Random state      : {RANDOM_STATE}")
print(f"Device            : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")

print()
print("✓ Configuration completed.")

NOTEBOOK 8 — CONFIGURATION

Project directory : E:\ML_Project
Training images   : E:\ML_Project\datasets\bdd100k_original_yolo\train\images
Oracle dataset    : E:\ML_Project\results\oracle_labels\policy_train_00000_70000.csv
Output directory  : E:\ML_Project\results\final_analysis\cnn_policy
Model directory   : E:\ML_Project\results\final_analysis\cnn_policy\models

Actions:
0 : Identity
1 : CLAHE
2 : Gamma
3 : Dehazing
4 : Denoising

Image size        : 224 × 224
Batch size        : 128
Workers           : 4
Train ratio       : 0.8
Random state      : 42
Device            : cuda
GPU               : NVIDIA GeForce RTX 5060 Ti

✓ Configuration completed.


Oracle action and margin calculation.

This uses the existing five reward columns only. Nothing is modified or regenerated.

In [5]:
# ============================================================
# CELL 4 — ORACLE ACTION AND MARGIN CALCULATION
# ============================================================

print("=" * 80)
print("CALCULATING ORACLE ACTIONS AND MARGINS")
print("=" * 80)

reward_matrix = oracle_df[
    REWARD_COLUMNS
].to_numpy(
    dtype=np.float32
)

oracle_actions = np.argmax(
    reward_matrix,
    axis=1
)

sorted_rewards = np.sort(
    reward_matrix,
    axis=1
)

top1_rewards = sorted_rewards[:, -1]
top2_rewards = sorted_rewards[:, -2]

oracle_margins = (
    top1_rewards -
    top2_rewards
)

oracle_df["oracle_action"] = oracle_actions
oracle_df["oracle_top1_reward"] = top1_rewards
oracle_df["oracle_top2_reward"] = top2_rewards
oracle_df["oracle_margin"] = oracle_margins

print()
print("=" * 80)
print("ORACLE ACTION DISTRIBUTION")
print("=" * 80)

for action_id, action_name in ID_TO_ACTION.items():

    count = int(
        np.sum(
            oracle_actions == action_id
        )
    )

    percentage = (
        count /
        len(oracle_actions)
    ) * 100

    print(
        f"{action_id} ({action_name:<10}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )

print()
print("=" * 80)
print("ORACLE MARGIN STATISTICS")
print("=" * 80)

print()
print(
    f"Mean margin   : "
    f"{oracle_margins.mean():.6f}"
)

print(
    f"Median margin : "
    f"{np.median(oracle_margins):.6f}"
)

print(
    f"Minimum       : "
    f"{oracle_margins.min():.6f}"
)

print(
    f"Maximum       : "
    f"{oracle_margins.max():.6f}"
)

print()
print("=" * 80)
print("MARGIN DISTRIBUTION")
print("=" * 80)

margin_thresholds = [
    0.001,
    0.005,
    0.010,
    0.020,
    0.050,
    0.100
]

print()

for threshold in margin_thresholds:

    count = int(
        np.sum(
            oracle_margins < threshold
        )
    )

    percentage = (
        count /
        len(oracle_margins)
    ) * 100

    print(
        f"Margin < {threshold:.3f} : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )

print()
print("=" * 80)
print("ORACLE TARGET VALIDATION")
print("=" * 80)

if len(oracle_df) != 70000:
    raise ValueError(
        "Oracle row count changed unexpectedly."
    )

if np.isnan(oracle_margins).any():
    raise ValueError(
        "NaN values found in Oracle margins."
    )

if np.isinf(oracle_margins).any():
    raise ValueError(
        "Infinite values found in Oracle margins."
    )

if np.any(oracle_margins < 0):
    raise ValueError(
        "Negative Oracle margins detected."
    )

print()
print("✓ Oracle actions calculated.")
print("✓ Top-1 rewards calculated.")
print("✓ Top-2 rewards calculated.")
print("✓ Oracle margins calculated.")
print("✓ All 70,000 samples retained.")
print("✓ No reward values modified.")
print("✓ No Oracle labels regenerated.")

print()
print("=" * 80)
print("CELL 4 COMPLETED")
print("=" * 80)

CALCULATING ORACLE ACTIONS AND MARGINS

ORACLE ACTION DISTRIBUTION
0 (Identity  ) : 25,379 ( 36.26%)
1 (CLAHE     ) : 14,870 ( 21.24%)
2 (Gamma     ) :  8,646 ( 12.35%)
3 (Dehazing  ) : 10,182 ( 14.55%)
4 (Denoising ) : 10,923 ( 15.60%)

ORACLE MARGIN STATISTICS

Mean margin   : 0.027634
Median margin : 0.017310
Minimum       : 0.000000
Maximum       : 0.571429

MARGIN DISTRIBUTION

Margin < 0.001 : 24,651 ( 35.22%)
Margin < 0.005 : 25,819 ( 36.88%)
Margin < 0.010 : 28,180 ( 40.26%)
Margin < 0.020 : 37,716 ( 53.88%)
Margin < 0.050 : 56,842 ( 81.20%)
Margin < 0.100 : 66,837 ( 95.48%)

ORACLE TARGET VALIDATION

✓ Oracle actions calculated.
✓ Top-1 rewards calculated.
✓ Top-2 rewards calculated.
✓ Oracle margins calculated.
✓ All 70,000 samples retained.
✓ No reward values modified.
✓ No Oracle labels regenerated.

CELL 4 COMPLETED


Verify and Match All 70K Images

In [6]:
# ============================================================
# CELL 5 — VERIFY AND MATCH 70K TRAINING IMAGES
# ============================================================

print("=" * 80)
print("VERIFYING 70K ORACLE IMAGE KEYS AGAINST TRAINING IMAGES")
print("=" * 80)

if not TRAIN_IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Training image directory not found:\n{TRAIN_IMAGE_DIR}"
    )

print()
print(f"Training image directory : {TRAIN_IMAGE_DIR}")

image_files = list(
    TRAIN_IMAGE_DIR.glob("*.jpg")
)

print()
print("=" * 80)
print("TRAINING IMAGE COUNT")
print("=" * 80)

print()
print(f"Image files found : {len(image_files):,}")

if len(image_files) != 70000:
    raise ValueError(
        f"Expected exactly 70,000 training images, "
        f"but found {len(image_files):,}."
    )

oracle_image_names = (
    oracle_df["image"]
    .astype(str)
    .str.strip()
)

dataset_image_names = {
    image_file.name
    for image_file in image_files
}

oracle_keys = set(
    oracle_image_names
)

common_keys = (
    oracle_keys &
    dataset_image_names
)

oracle_only = (
    oracle_keys -
    dataset_image_names
)

dataset_only = (
    dataset_image_names -
    oracle_keys
)

print()
print("=" * 80)
print("IMAGE KEY ALIGNMENT")
print("=" * 80)

print()
print(f"Oracle images : {len(oracle_keys):,}")
print(f"Dataset images: {len(dataset_image_names):,}")
print(f"Common        : {len(common_keys):,}")
print(f"Oracle-only   : {len(oracle_only):,}")
print(f"Dataset-only  : {len(dataset_only):,}")

if len(oracle_keys) != 70000:
    raise ValueError(
        "Oracle image identifiers are not unique."
    )

if len(dataset_image_names) != 70000:
    raise ValueError(
        "Training image filenames are not unique."
    )

if len(common_keys) != 70000:
    sample_missing = sorted(
        oracle_only
    )[:10]

    raise ValueError(
        "Not all 70,000 Oracle images were matched.\n\n"
        f"Missing count : {len(oracle_only):,}\n"
        f"Examples      : {sample_missing}"
    )

print()
print("✓ All 70,000 Oracle image names matched.")

oracle_df["image_path"] = (
    oracle_df["image"]
    .astype(str)
    .str.strip()
    .map(
        lambda x:
        str(
            TRAIN_IMAGE_DIR / x
        )
    )
)

missing_paths = (
    ~oracle_df["image_path"]
    .map(
        lambda x:
        Path(x).exists()
    )
)

missing_path_count = int(
    missing_paths.sum()
)

print()
print("=" * 80)
print("IMAGE PATH VALIDATION")
print("=" * 80)

print()
print(
    f"Expected image paths : "
    f"{len(oracle_df):,}"
)

print(
    f"Missing image paths  : "
    f"{missing_path_count:,}"
)

if missing_path_count != 0:
    raise FileNotFoundError(
        "One or more Oracle image paths do not exist."
    )

print()
print("✓ All 70,000 image paths exist.")

print()
print("=" * 80)
print("VERIFYING IMAGE READABILITY")
print("=" * 80)

sample_size = min(
    100,
    len(oracle_df)
)

sample_indices = np.linspace(
    0,
    len(oracle_df) - 1,
    sample_size,
    dtype=int
)

failed_reads = []

for index in sample_indices:

    image_path = Path(
        oracle_df.iloc[index]["image_path"]
    )

    try:
        with Image.open(image_path) as image:
            image.verify()

    except Exception as error:
        failed_reads.append(
            (
                str(image_path),
                str(error)
            )
        )

print()
print(f"Images sampled : {sample_size:,}")
print(f"Failed reads   : {len(failed_reads):,}")

if failed_reads:
    print()
    print("Examples:")

    for path, error in failed_reads[:5]:
        print(path)
        print(error)

    raise ValueError(
        "Unreadable images detected."
    )

print()
print("✓ Sampled training images are readable.")

print()
print("=" * 80)
print("70K IMAGE ALIGNMENT COMPLETED")
print("=" * 80)

print()
print("✓ 70,000 Oracle image keys verified.")
print("✓ All 70,000 Oracle images matched.")
print("✓ All image paths verified.")
print("✓ Image readability check passed.")
print("✓ No Oracle values modified.")
print("✓ No YOLO inference performed.")

VERIFYING 70K ORACLE IMAGE KEYS AGAINST TRAINING IMAGES

Training image directory : E:\ML_Project\datasets\bdd100k_original_yolo\train\images

TRAINING IMAGE COUNT

Image files found : 70,000

IMAGE KEY ALIGNMENT

Oracle images : 70,000
Dataset images: 70,000
Common        : 70,000
Oracle-only   : 0
Dataset-only  : 0

✓ All 70,000 Oracle image names matched.

IMAGE PATH VALIDATION

Expected image paths : 70,000
Missing image paths  : 0

✓ All 70,000 image paths exist.

VERIFYING IMAGE READABILITY

Images sampled : 100
Failed reads   : 0

✓ Sampled training images are readable.

70K IMAGE ALIGNMENT COMPLETED

✓ 70,000 Oracle image keys verified.
✓ All 70,000 Oracle images matched.
✓ All image paths verified.
✓ Image readability check passed.
✓ No Oracle values modified.
✓ No YOLO inference performed.


create the 80/20 policy train/internal-validation split.

In [7]:
# ============================================================
# CELL 6 — CREATE POLICY TRAIN / INTERNAL VALIDATION SPLIT
# ============================================================

print("=" * 80)
print("CREATING POLICY TRAIN / INTERNAL VALIDATION SPLIT")
print("=" * 80)

if len(oracle_df) != 70000:
    raise ValueError(
        f"Expected 70,000 samples, found {len(oracle_df):,}."
    )

if "oracle_action" not in oracle_df.columns:
    raise KeyError(
        "oracle_action column not found. Run Cell 4 first."
    )

print()
print("Original dataset")
print("----------------")
print(f"Total samples : {len(oracle_df):,}")

class_counts = (
    oracle_df["oracle_action"]
    .value_counts()
    .sort_index()
)

print()
print("=" * 80)
print("ORIGINAL ORACLE ACTION DISTRIBUTION")
print("=" * 80)

print()

for action_id, action_name in ID_TO_ACTION.items():

    count = int(
        class_counts.get(
            action_id,
            0
        )
    )

    percentage = (
        count /
        len(oracle_df)
    ) * 100

    print(
        f"{action_id} ({action_name:<10}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )

train_df, internal_val_df = train_test_split(
    oracle_df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=oracle_df["oracle_action"]
)

train_df = train_df.reset_index(
    drop=True
)

internal_val_df = internal_val_df.reset_index(
    drop=True
)

print()
print("=" * 80)
print("SPLIT RESULT")
print("=" * 80)

print()
print(
    f"Total samples      : "
    f"{len(oracle_df):,}"
)

print(
    f"Training samples   : "
    f"{len(train_df):,}"
)

print(
    f"Internal validation: "
    f"{len(internal_val_df):,}"
)

print(
    f"Training ratio     : "
    f"{len(train_df) / len(oracle_df) * 100:.2f}%"
)

print(
    f"Validation ratio   : "
    f"{len(internal_val_df) / len(oracle_df) * 100:.2f}%"
)

print()
print("=" * 80)
print("TRAINING CLASS DISTRIBUTION")
print("=" * 80)

print()

train_counts = (
    train_df["oracle_action"]
    .value_counts()
    .sort_index()
)

for action_id, action_name in ID_TO_ACTION.items():

    count = int(
        train_counts.get(
            action_id,
            0
        )
    )

    percentage = (
        count /
        len(train_df)
    ) * 100

    print(
        f"{action_id} ({action_name:<10}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )

print()
print("=" * 80)
print("INTERNAL VALIDATION CLASS DISTRIBUTION")
print("=" * 80)

print()

val_counts = (
    internal_val_df["oracle_action"]
    .value_counts()
    .sort_index()
)

for action_id, action_name in ID_TO_ACTION.items():

    count = int(
        val_counts.get(
            action_id,
            0
        )
    )

    percentage = (
        count /
        len(internal_val_df)
    ) * 100

    print(
        f"{action_id} ({action_name:<10}) : "
        f"{count:6,} "
        f"({percentage:6.2f}%)"
    )

train_images = set(
    train_df["image"].astype(str)
)

val_images = set(
    internal_val_df["image"].astype(str)
)

overlap = (
    train_images &
    val_images
)

print()
print("=" * 80)
print("IMAGE LEAKAGE CHECK")
print("=" * 80)

print()
print(
    f"Training image keys   : "
    f"{len(train_images):,}"
)

print(
    f"Validation image keys : "
    f"{len(val_images):,}"
)

print(
    f"Overlap               : "
    f"{len(overlap):,}"
)

if len(overlap) != 0:
    raise ValueError(
        "Image leakage detected between "
        "training and internal validation sets."
    )

print()
print("✓ No image overlap detected.")

if (
    len(train_df) +
    len(internal_val_df)
) != 70000:

    raise ValueError(
        "Train + validation samples do not equal 70,000."
    )

print()
print("=" * 80)
print("10K FINAL TEST SET PROTECTION")
print("=" * 80)

print()
print(
    "✓ The existing 10K final evaluation dataset "
    "is not loaded or used in this split."
)

print()
print("=" * 80)
print("CELL 6 COMPLETED")
print("=" * 80)

print()
print("✓ 56,000 training samples.")
print("✓ 14,000 internal validation samples.")
print("✓ Stratified by Oracle action.")
print("✓ No train/validation image overlap.")
print("✓ 10K final test set remains untouched.")

CREATING POLICY TRAIN / INTERNAL VALIDATION SPLIT

Original dataset
----------------
Total samples : 70,000

ORIGINAL ORACLE ACTION DISTRIBUTION

0 (Identity  ) : 25,379 ( 36.26%)
1 (CLAHE     ) : 14,870 ( 21.24%)
2 (Gamma     ) :  8,646 ( 12.35%)
3 (Dehazing  ) : 10,182 ( 14.55%)
4 (Denoising ) : 10,923 ( 15.60%)

SPLIT RESULT

Total samples      : 70,000
Training samples   : 56,000
Internal validation: 14,000
Training ratio     : 80.00%
Validation ratio   : 20.00%

TRAINING CLASS DISTRIBUTION

0 (Identity  ) : 20,303 ( 36.26%)
1 (CLAHE     ) : 11,896 ( 21.24%)
2 (Gamma     ) :  6,917 ( 12.35%)
3 (Dehazing  ) :  8,146 ( 14.55%)
4 (Denoising ) :  8,738 ( 15.60%)

INTERNAL VALIDATION CLASS DISTRIBUTION

0 (Identity  ) :  5,076 ( 36.26%)
1 (CLAHE     ) :  2,974 ( 21.24%)
2 (Gamma     ) :  1,729 ( 12.35%)
3 (Dehazing  ) :  2,036 ( 14.54%)
4 (Denoising ) :  2,185 ( 15.61%)

IMAGE LEAKAGE CHECK

Training image keys   : 56,000
Validation image keys : 14,000
Overlap               : 0

✓ No im

Build PyTorch Dataset & DataLoaders

In [11]:
# ============================================================
# CELL 7 — PYTORCH DATASET AND DATALOADERS
# ============================================================

print("=" * 80)
print("BUILDING PYTORCH IMAGE POLICY DATASETS")
print("=" * 80)

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

val_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


class ImagePolicyDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.transform = transform

    def __len__(self):

        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform is not None:

            image = self.transform(
                image
            )

        action = torch.tensor(
            int(row["oracle_action"]),
            dtype=torch.long
        )

        margin = torch.tensor(
            float(row["oracle_margin"]),
            dtype=torch.float32
        )

        rewards = torch.tensor(
            row[REWARD_COLUMNS].to_numpy(
                dtype=np.float32
            ),
            dtype=torch.float32
        )

        return {
            "image": image,
            "action": action,
            "margin": margin,
            "rewards": rewards,
            "image_name": row["image"]
        }


train_dataset = ImagePolicyDataset(
    dataframe=train_df,
    transform=train_transform
)

internal_val_dataset = ImagePolicyDataset(
    dataframe=internal_val_df,
    transform=val_transform
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)

internal_val_loader = DataLoader(
    internal_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False
)


print()
print("=" * 80)
print("DATALOADER INFORMATION")
print("=" * 80)

print()
print(
    f"Training dataset   : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation dataset : "
    f"{len(internal_val_dataset):,}"
)

print(
    f"Training batches   : "
    f"{len(train_loader):,}"
)

print(
    f"Validation batches : "
    f"{len(internal_val_loader):,}"
)

print(
    "DataLoader workers : 0"
)


print()
print("=" * 80)
print("BENCHMARKING TRAINING DATALOADER")
print("=" * 80)

loader_iterator = iter(
    train_loader
)

benchmark_batches = 5

benchmark_start = time.time()

for batch_index in range(
    benchmark_batches
):

    batch = next(
        loader_iterator
    )

benchmark_time = (
    time.time() -
    benchmark_start
)

images_processed = (
    benchmark_batches *
    BATCH_SIZE
)

images_per_second = (
    images_processed /
    benchmark_time
)

estimated_epoch_time = (
    len(train_dataset) /
    images_per_second
)


print()
print(
    f"Batches tested     : "
    f"{benchmark_batches}"
)

print(
    f"Images processed   : "
    f"{images_processed:,}"
)

print(
    f"Benchmark time     : "
    f"{benchmark_time:.3f} sec"
)

print(
    f"Images/sec         : "
    f"{images_per_second:.2f}"
)

print(
    f"Estimated epoch    : "
    f"{estimated_epoch_time / 60:.2f} min"
)


print()
print("=" * 80)
print("TESTING BATCH STRUCTURE")
print("=" * 80)

sample_batch = batch

print()
print(
    f"Image tensor shape : "
    f"{tuple(sample_batch['image'].shape)}"
)

print(
    f"Action shape       : "
    f"{tuple(sample_batch['action'].shape)}"
)

print(
    f"Margin shape       : "
    f"{tuple(sample_batch['margin'].shape)}"
)

print(
    f"Reward shape       : "
    f"{tuple(sample_batch['rewards'].shape)}"
)

if tuple(
    sample_batch["image"].shape
) != (
    BATCH_SIZE,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
):

    raise ValueError(
        "Unexpected image batch shape."
    )

if sample_batch["action"].shape[0] != BATCH_SIZE:

    raise ValueError(
        "Unexpected action batch size."
    )

if sample_batch["rewards"].shape[1] != 5:

    raise ValueError(
        "Expected five reward values."
    )

if torch.isnan(
    sample_batch["image"]
).any():

    raise ValueError(
        "NaN values found in image batch."
    )

if torch.isinf(
    sample_batch["image"]
).any():

    raise ValueError(
        "Infinite values found in image batch."
    )


print()
print("=" * 80)
print("CELL 7 COMPLETED")
print("=" * 80)

print()
print("✓ 56,000 training images ready.")
print("✓ 14,000 validation images ready.")
print("✓ Single-process DataLoader used.")
print("✓ 5-batch throughput benchmark completed.")
print("✓ Image tensors verified.")
print("✓ Oracle targets verified.")

BUILDING PYTORCH IMAGE POLICY DATASETS

DATALOADER INFORMATION

Training dataset   : 56,000
Validation dataset : 14,000
Training batches   : 438
Validation batches : 110
DataLoader workers : 0

BENCHMARKING TRAINING DATALOADER

Batches tested     : 5
Images processed   : 640
Benchmark time     : 5.171 sec
Images/sec         : 123.76
Estimated epoch    : 7.54 min

TESTING BATCH STRUCTURE

Image tensor shape : (128, 3, 224, 224)
Action shape       : (128,)
Margin shape       : (128,)
Reward shape       : (128, 5)

CELL 7 COMPLETED

✓ 56,000 training images ready.
✓ 14,000 validation images ready.
✓ Single-process DataLoader used.
✓ 5-batch throughput benchmark completed.
✓ Image tensors verified.
✓ Oracle targets verified.


In [ ]:
Tried on the single image to analyze the work

In [9]:
IMAGE_SIZE = 224
IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

TRAIN_IMAGE_DIR = Path(
    r"E:\ML_Project\datasets\bdd100k_original_yolo\train\images"
)

test_path = next(
    TRAIN_IMAGE_DIR.glob("*.jpg")
)

print("=" * 80)
print("SINGLE IMAGE TRANSFORM DIAGNOSTIC")
print("=" * 80)

print()
print("Test image:")
print(test_path)

start = time.time()

image = Image.open(
    test_path
).convert("RGB")

jpeg_time = time.time() - start

print()
print(
    f"JPEG load time : "
    f"{jpeg_time:.4f} sec"
)

start = time.time()

tensor = transform(
    image
)

transform_time = time.time() - start

print(
    f"Transform time : "
    f"{transform_time:.4f} sec"
)

total_time = (
    jpeg_time +
    transform_time
)

print(
    f"Total time     : "
    f"{total_time:.4f} sec"
)

print(
    f"Final shape    : "
    f"{tuple(tensor.shape)}"
)

print(
    f"Final dtype    : "
    f"{tensor.dtype}"
)

print()
print("=" * 80)
print("DIAGNOSTIC COMPLETED")
print("=" * 80)

SINGLE IMAGE TRANSFORM DIAGNOSTIC

Test image:
E:\ML_Project\datasets\bdd100k_original_yolo\train\images\0000f77c-6257be58.jpg

JPEG load time : 0.0035 sec
Transform time : 0.0060 sec
Total time     : 0.0095 sec
Final shape    : (3, 224, 224)
Final dtype    : torch.float32

DIAGNOSTIC COMPLETED


In [ ]:
Define the CNN Policy 

In [12]:
print("=" * 80)
print("BUILDING CNN POLICY MODELS")
print("=" * 80)


class CNNPolicyBaseline(nn.Module):

    def __init__(
        self,
        num_classes=5
    ):

        super().__init__()

        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        feature_dim = backbone.fc.in_features

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.policy_head = nn.Sequential(
            nn.Linear(
                feature_dim,
                256
            ),
            nn.ReLU(),
            nn.Dropout(
                p=0.30
            ),
            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(
        self,
        x
    ):

        features = self.backbone(x)

        logits = self.policy_head(
            features
        )

        return logits


class ConvBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.SiLU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(
                out_channels
            ),
            nn.SiLU()
        )

    def forward(
        self,
        x
    ):

        return self.block(x)


class GlobalBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(
                3,
                32,
                stride=2
            ),
            ConvBlock(
                32,
                64,
                stride=2
            ),
            ConvBlock(
                64,
                128,
                stride=2
            ),
            ConvBlock(
                128,
                256,
                stride=2
            )
        )

        self.pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

    def forward(
        self,
        x
    ):

        x = self.features(x)

        x = self.pool(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        return x


class LocalBranch(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(
                3,
                32,
                stride=2
            ),
            ConvBlock(
                32,
                64,
                stride=2
            ),
            ConvBlock(
                64,
                128,
                stride=2
            )
        )

        self.pool = nn.AdaptiveAvgPool2d(
            (2, 2)
        )

    def forward(
        self,
        x
    ):

        x = self.features(x)

        x = self.pool(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        return x


class FeatureGating(nn.Module):

    def __init__(
        self,
        feature_dim
    ):

        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(
                feature_dim,
                feature_dim
            ),
            nn.Sigmoid()
        )

    def forward(
        self,
        x
    ):

        gate = self.gate(x)

        return x * gate


class GlobalLocalCNNPolicy(nn.Module):

    def __init__(
        self,
        num_classes=5
    ):

        super().__init__()

        self.global_branch = GlobalBranch()

        self.local_branch = LocalBranch()

        global_dim = 256

        local_dim = 128 * 2 * 2

        fusion_dim = (
            global_dim +
            local_dim
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                fusion_dim,
                512
            ),
            nn.BatchNorm1d(
                512
            ),
            nn.SiLU(),
            nn.Dropout(
                p=0.30
            )
        )

        self.gating = FeatureGating(
            512
        )

        self.policy_head = nn.Sequential(
            nn.Linear(
                512,
                256
            ),
            nn.SiLU(),
            nn.Dropout(
                p=0.30
            ),
            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(
        self,
        x
    ):

        global_features = (
            self.global_branch(x)
        )

        local_features = (
            self.local_branch(x)
        )

        fused_features = torch.cat(
            [
                global_features,
                local_features
            ],
            dim=1
        )

        fused_features = self.fusion(
            fused_features
        )

        gated_features = self.gating(
            fused_features
        )

        logits = self.policy_head(
            gated_features
        )

        return logits


baseline_model = CNNPolicyBaseline(
    num_classes=5
).to(DEVICE)

proposed_model = GlobalLocalCNNPolicy(
    num_classes=5
).to(DEVICE)

print()
print("=" * 80)
print("MODEL VERIFICATION")
print("=" * 80)

print()
print(
    f"Baseline model device : "
    f"{next(baseline_model.parameters()).device}"
)

print(
    f"Proposed model device : "
    f"{next(proposed_model.parameters()).device}"
)

test_images = sample_batch[
    "image"
][:4].to(
    DEVICE,
    non_blocking=True
)

with torch.no_grad():

    baseline_output = baseline_model(
        test_images
    )

    proposed_output = proposed_model(
        test_images
    )

print()
print(
    f"Test input shape      : "
    f"{tuple(test_images.shape)}"
)

print(
    f"Baseline output shape : "
    f"{tuple(baseline_output.shape)}"
)

print(
    f"Proposed output shape : "
    f"{tuple(proposed_output.shape)}"
)

if baseline_output.shape != (4, 5):

    raise ValueError(
        "Baseline model output shape is incorrect."
    )

if proposed_output.shape != (4, 5):

    raise ValueError(
        "Proposed model output shape is incorrect."
    )

if torch.isnan(
    baseline_output
).any():

    raise ValueError(
        "NaN detected in baseline output."
    )

if torch.isnan(
    proposed_output
).any():

    raise ValueError(
        "NaN detected in proposed model output."
    )

baseline_parameters = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
)

proposed_parameters = sum(
    parameter.numel()
    for parameter in proposed_model.parameters()
)

print()
print(
    f"Baseline parameters : "
    f"{baseline_parameters:,}"
)

print(
    f"Proposed parameters : "
    f"{proposed_parameters:,}"
)

print()
print("=" * 80)
print("CELL 8 COMPLETED")
print("=" * 80)

print()
print("✓ Standard CNN baseline created.")
print("✓ Global branch created.")
print("✓ Local branch created.")
print("✓ Feature fusion created.")
print("✓ Feature gating created.")
print("✓ Proposed CNN policy created.")
print("✓ Both models moved to CUDA when available.")
print("✓ Forward pass verified.")
print("✓ Five-action output verified.")

BUILDING CNN POLICY MODELS

MODEL VERIFICATION

Baseline model device : cuda:0
Proposed model device : cuda:0

Test input shape      : (4, 3, 224, 224)
Baseline output shape : (4, 5)
Proposed output shape : (4, 5)

Baseline parameters : 11,309,125
Proposed parameters : 2,250,693

CELL 8 COMPLETED

✓ Standard CNN baseline created.
✓ Global branch created.
✓ Local branch created.
✓ Feature fusion created.
✓ Feature gating created.
✓ Proposed CNN policy created.
✓ Both models moved to CUDA when available.
✓ Forward pass verified.
✓ Five-action output verified.


Loss Functions

We will use:

L=L
CE
	​

+λL
Rank
	​

Cross-Entropy: learns the Oracle's preferred action.
Ranking loss: encourages the predicted score of the Oracle action to exceed the alternatives.
Margin weighting: higher-confidence Oracle decisions contribute more strongly.

In [13]:
print("=" * 80)
print("BUILDING POLICY LOSS FUNCTIONS")
print("=" * 80)


class MarginWeightedCrossEntropy(nn.Module):

    def __init__(
        self,
        minimum_weight=0.25
    ):

        super().__init__()

        self.minimum_weight = minimum_weight

    def forward(
        self,
        logits,
        targets,
        margins
    ):

        losses = F.cross_entropy(
            logits,
            targets,
            reduction="none"
        )

        normalized_margins = torch.clamp(
            margins,
            min=0.0
        )

        weights = (
            self.minimum_weight +
            normalized_margins
        )

        weights = weights / (
            weights.mean() + 1e-8
        )

        weighted_loss = (
            losses * weights
        ).mean()

        return weighted_loss


class OracleRankingLoss(nn.Module):

    def __init__(
        self,
        margin_scale=1.0
    ):

        super().__init__()

        self.margin_scale = margin_scale

    def forward(
        self,
        logits,
        targets,
        margins
    ):

        batch_size = logits.size(0)

        target_scores = logits[
            torch.arange(
                batch_size,
                device=logits.device
            ),
            targets
        ]

        mask = torch.ones_like(
            logits,
            dtype=torch.bool
        )

        mask[
            torch.arange(
                batch_size,
                device=logits.device
            ),
            targets
        ] = False

        alternative_scores = (
            logits[mask]
            .view(
                batch_size,
                -1
            )
        )

        target_scores = (
            target_scores
            .unsqueeze(1)
        )

        required_margin = (
            torch.clamp(
                margins,
                min=0.0
            )
            .unsqueeze(1)
            .detach()
            * self.margin_scale
        )

        ranking_loss = F.relu(
            alternative_scores -
            target_scores +
            required_margin
        )

        return ranking_loss.mean()


class CombinedPolicyLoss(nn.Module):

    def __init__(
        self,
        ranking_weight=0.25,
        minimum_weight=0.25,
        margin_scale=1.0
    ):

        super().__init__()

        self.ce_loss = (
            MarginWeightedCrossEntropy(
                minimum_weight=minimum_weight
            )
        )

        self.ranking_loss = (
            OracleRankingLoss(
                margin_scale=margin_scale
            )
        )

        self.ranking_weight = (
            ranking_weight
        )

    def forward(
        self,
        logits,
        targets,
        margins
    ):

        ce = self.ce_loss(
            logits,
            targets,
            margins
        )

        ranking = self.ranking_loss(
            logits,
            targets,
            margins
        )

        total = (
            ce +
            self.ranking_weight *
            ranking
        )

        return (
            total,
            ce,
            ranking
        )


policy_loss = CombinedPolicyLoss(
    ranking_weight=0.25,
    minimum_weight=0.25,
    margin_scale=1.0
).to(DEVICE)


print()
print("=" * 80)
print("LOSS CONFIGURATION")
print("=" * 80)

print()
print("Classification loss : Margin-weighted Cross-Entropy")
print("Ranking loss        : Oracle Ranking Loss")
print("Ranking weight      : 0.25")
print("Minimum CE weight   : 0.25")
print("Margin scale        : 1.0")

print()
print("=" * 80)
print("TESTING LOSS FUNCTIONS")
print("=" * 80)

test_targets = sample_batch[
    "action"
][:4].to(
    DEVICE
)

test_margins = sample_batch[
    "margin"
][:4].to(
    DEVICE
)

with torch.no_grad():

    test_logits = proposed_model(
        test_images
    )

test_total, test_ce, test_ranking = (
    policy_loss(
        test_logits,
        test_targets,
        test_margins
    )
)

print()
print(
    f"Total loss   : "
    f"{test_total.item():.6f}"
)

print(
    f"CE loss      : "
    f"{test_ce.item():.6f}"
)

print(
    f"Ranking loss : "
    f"{test_ranking.item():.6f}"
)

if not torch.isfinite(
    test_total
):

    raise ValueError(
        "Invalid total loss."
    )

if not torch.isfinite(
    test_ce
):

    raise ValueError(
        "Invalid classification loss."
    )

if not torch.isfinite(
    test_ranking
):

    raise ValueError(
        "Invalid ranking loss."
    )

print()
print("=" * 80)
print("CELL 9 COMPLETED")
print("=" * 80)

print()
print("✓ Margin-weighted classification loss verified.")
print("✓ Ranking loss verified.")
print("✓ Combined policy loss verified.")
print("✓ No NaN values.")
print("✓ No Inf values.")
print("✓ Loss computation runs on CUDA.")

BUILDING POLICY LOSS FUNCTIONS

LOSS CONFIGURATION

Classification loss : Margin-weighted Cross-Entropy
Ranking loss        : Oracle Ranking Loss
Ranking weight      : 0.25
Minimum CE weight   : 0.25
Margin scale        : 1.0

TESTING LOSS FUNCTIONS

Total loss   : 1.668473
CE loss      : 1.633575
Ranking loss : 0.139591

CELL 9 COMPLETED

✓ Margin-weighted classification loss verified.
✓ Ranking loss verified.
✓ Combined policy loss verified.
✓ No NaN values.
✓ No Inf values.
✓ Loss computation runs on CUDA.


Setting Up the Standard CNN Baseline Training

In [14]:
print("=" * 80)
print("SETTING UP STANDARD CNN BASELINE TRAINING")
print("=" * 80)

BASELINE_EPOCHS = 20

BASELINE_LR = 1e-4

BASELINE_WEIGHT_DECAY = 1e-4

BASELINE_PATIENCE = 5

baseline_criterion = nn.CrossEntropyLoss()

baseline_optimizer = torch.optim.AdamW(
    baseline_model.parameters(),
    lr=BASELINE_LR,
    weight_decay=BASELINE_WEIGHT_DECAY
)

baseline_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    baseline_optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

baseline_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)

best_baseline_accuracy = -np.inf

best_baseline_state = None

baseline_history = []

baseline_checkpoint = (
    MODEL_DIR /
    "cnn_baseline_best.pt"
)

print()
print("=" * 80)
print("BASELINE TRAINING CONFIGURATION")
print("=" * 80)

print()
print(
    f"Model             : ResNet18"
)

print(
    f"Training samples  : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation samples: "
    f"{len(internal_val_dataset):,}"
)

print(
    f"Epochs            : "
    f"{BASELINE_EPOCHS}"
)

print(
    f"Learning rate     : "
    f"{BASELINE_LR}"
)

print(
    f"Weight decay      : "
    f"{BASELINE_WEIGHT_DECAY}"
)

print(
    f"Batch size        : "
    f"{BATCH_SIZE}"
)

print(
    f"Optimizer         : AdamW"
)

print(
    f"Loss              : Cross-Entropy"
)

print(
    f"Scheduler         : ReduceLROnPlateau"
)

print(
    f"Early stopping    : "
    f"{BASELINE_PATIENCE} epochs"
)

print(
    f"Device            : "
    f"{DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU               : "
        f"{torch.cuda.get_device_name(0)}"
    )

print()
print("=" * 80)
print("TESTING BASELINE OPTIMIZER")
print("=" * 80)

baseline_optimizer.zero_grad(
    set_to_none=True
)

test_images = sample_batch[
    "image"
].to(
    DEVICE,
    non_blocking=True
)

test_targets = sample_batch[
    "action"
].to(
    DEVICE,
    non_blocking=True
)

with torch.amp.autocast(
    device_type="cuda",
    enabled=torch.cuda.is_available()
):

    test_logits = baseline_model(
        test_images
    )

    test_loss = baseline_criterion(
        test_logits,
        test_targets
    )

print()
print(
    f"Test CE loss : "
    f"{test_loss.item():.6f}"
)

if not torch.isfinite(
    test_loss
):

    raise ValueError(
        "Baseline loss is not finite."
    )

print()
print("=" * 80)
print("CELL 10 COMPLETED")
print("=" * 80)

print()
print("✓ Standard CNN baseline optimizer created.")
print("✓ Cross-Entropy loss configured.")
print("✓ AdamW configured.")
print("✓ Learning-rate scheduler configured.")
print("✓ CUDA AMP configured.")
print("✓ Best-model checkpoint path configured.")
print("✓ Baseline forward/loss test passed.")

SETTING UP STANDARD CNN BASELINE TRAINING

BASELINE TRAINING CONFIGURATION

Model             : ResNet18
Training samples  : 56,000
Validation samples: 14,000
Epochs            : 20
Learning rate     : 0.0001
Weight decay      : 0.0001
Batch size        : 128
Optimizer         : AdamW
Loss              : Cross-Entropy
Scheduler         : ReduceLROnPlateau
Early stopping    : 5 epochs
Device            : cuda
GPU               : NVIDIA GeForce RTX 5060 Ti

TESTING BASELINE OPTIMIZER

Test CE loss : 1.745583

CELL 10 COMPLETED

✓ Standard CNN baseline optimizer created.
✓ Cross-Entropy loss configured.
✓ AdamW configured.
✓ Learning-rate scheduler configured.
✓ CUDA AMP configured.
✓ Best-model checkpoint path configured.
✓ Baseline forward/loss test passed.


Train the Standard CNN Baseline

In [15]:
print("=" * 80)
print("TRAINING STANDARD CNN BASELINE")
print("=" * 80)

def train_baseline_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["action"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == targets
        ).sum().item()

        total += images.size(0)

    epoch_loss = (
        running_loss /
        total
    )

    epoch_accuracy = (
        correct /
        total
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


@torch.no_grad()
def validate_baseline(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    total = 0

    all_predictions = []
    all_targets = []

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["action"].to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            logits = model(
                images
            )

            loss = criterion(
                logits,
                targets
            )

        running_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            targets.cpu()
        )

        total += images.size(0)

    validation_loss = (
        running_loss /
        total
    )

    all_predictions = torch.cat(
        all_predictions
    ).numpy()

    all_targets = torch.cat(
        all_targets
    ).numpy()

    validation_accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    validation_balanced_accuracy = (
        balanced_accuracy_score(
            all_targets,
            all_predictions
        )
    )

    validation_macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro"
    )

    return (
        validation_loss,
        validation_accuracy,
        validation_balanced_accuracy,
        validation_macro_f1
    )


best_baseline_accuracy = -np.inf

best_baseline_macro_f1 = -np.inf

best_baseline_state = None

best_baseline_epoch = 0

epochs_without_improvement = 0

baseline_history = []

training_start = time.time()

print()
print("=" * 80)
print("TRAINING")
print("=" * 80)

for epoch in range(
    1,
    BASELINE_EPOCHS + 1
):

    epoch_start = time.time()

    train_loss, train_accuracy = (
        train_baseline_epoch(
            baseline_model,
            train_loader,
            baseline_criterion,
            baseline_optimizer,
            baseline_scaler,
            DEVICE
        )
    )

    (
        validation_loss,
        validation_accuracy,
        validation_balanced_accuracy,
        validation_macro_f1
    ) = validate_baseline(
        baseline_model,
        internal_val_loader,
        baseline_criterion,
        DEVICE
    )

    baseline_scheduler.step(
        validation_macro_f1
    )

    current_lr = (
        baseline_optimizer
        .param_groups[0]["lr"]
    )

    epoch_time = (
        time.time() -
        epoch_start
    )

    baseline_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
            "validation_balanced_accuracy":
                validation_balanced_accuracy,
            "validation_macro_f1":
                validation_macro_f1,
            "learning_rate": current_lr,
            "epoch_time_seconds": epoch_time
        }
    )

    print(
        f"Epoch {epoch:02d}/{BASELINE_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy * 100:.2f}% | "
        f"Val Loss: {validation_loss:.4f} | "
        f"Val Acc: {validation_accuracy * 100:.2f}% | "
        f"Val Macro-F1: {validation_macro_f1:.4f} | "
        f"LR: {current_lr:.2e} | "
        f"Time: {epoch_time:.1f}s"
    )

    if validation_macro_f1 > best_baseline_macro_f1:

        best_baseline_macro_f1 = (
            validation_macro_f1
        )

        best_baseline_accuracy = (
            validation_accuracy
        )

        best_baseline_state = copy.deepcopy(
            baseline_model.state_dict()
        )

        best_baseline_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    baseline_model.state_dict(),
                "optimizer_state_dict":
                    baseline_optimizer.state_dict(),
                "scheduler_state_dict":
                    baseline_scheduler.state_dict(),
                "validation_accuracy":
                    validation_accuracy,
                "validation_macro_f1":
                    validation_macro_f1
            },
            baseline_checkpoint
        )

    else:

        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= BASELINE_PATIENCE
    ):

        print()
        print(
            f"Early stopping at epoch {epoch}."
        )

        break


training_time = (
    time.time() -
    training_start
)

if best_baseline_state is None:

    raise RuntimeError(
        "No valid baseline checkpoint was produced."
    )

baseline_model.load_state_dict(
    best_baseline_state
)

baseline_history_df = pd.DataFrame(
    baseline_history
)

history_file = (
    OUTPUT_DIR /
    "cnn_baseline_training_history.csv"
)

baseline_history_df.to_csv(
    history_file,
    index=False
)

print()
print("=" * 80)
print("STANDARD CNN BASELINE TRAINING COMPLETED")
print("=" * 80)

print()
print(
    f"Training time        : "
    f"{training_time / 60:.2f} minutes"
)

print(
    f"Best epoch           : "
    f"{best_baseline_epoch}"
)

print(
    f"Best validation acc  : "
    f"{best_baseline_accuracy * 100:.2f}%"
)

print(
    f"Best validation F1   : "
    f"{best_baseline_macro_f1:.4f}"
)

print()
print(
    f"Checkpoint:"
)

print(
    baseline_checkpoint
)

print()
print(
    f"Training history:"
)

print(
    history_file
)

print()
print("✓ Best model restored.")
print("✓ Best checkpoint saved.")
print("✓ Internal validation used during training.")
print("✓ 10K final test set was not used.")
print("✓ CNN baseline training completed.")

TRAINING STANDARD CNN BASELINE

TRAINING
Epoch 01/20 | Train Loss: 1.5324 | Train Acc: 35.81% | Val Loss: 1.5194 | Val Acc: 36.20% | Val Macro-F1: 0.1091 | LR: 1.00e-04 | Time: 680.5s
Epoch 02/20 | Train Loss: 1.5103 | Train Acc: 36.32% | Val Loss: 1.5252 | Val Acc: 36.23% | Val Macro-F1: 0.1074 | LR: 1.00e-04 | Time: 583.1s
Epoch 03/20 | Train Loss: 1.4880 | Train Acc: 36.66% | Val Loss: 1.5400 | Val Acc: 34.46% | Val Macro-F1: 0.1512 | LR: 1.00e-04 | Time: 436.0s
Epoch 04/20 | Train Loss: 1.4403 | Train Acc: 38.68% | Val Loss: 1.5601 | Val Acc: 32.83% | Val Macro-F1: 0.1789 | LR: 1.00e-04 | Time: 447.0s
Epoch 05/20 | Train Loss: 1.3340 | Train Acc: 45.00% | Val Loss: 1.6948 | Val Acc: 30.63% | Val Macro-F1: 0.1943 | LR: 1.00e-04 | Time: 418.1s
Epoch 06/20 | Train Loss: 1.1401 | Train Acc: 55.36% | Val Loss: 1.8682 | Val Acc: 28.57% | Val Macro-F1: 0.2013 | LR: 1.00e-04 | Time: 443.4s
Epoch 07/20 | Train Loss: 0.8829 | Train Acc: 67.10% | Val Loss: 2.0774 | Val Acc: 24.27% | Val Macro

CNN 20K FINAL TEST EVALUATION

In [1]:
BASE_DIR = Path(
    r"E:\ML_Project"
)

TEST_IMAGE_DIR = (
    BASE_DIR /
    "datasets" /
    "bdd100k_original_yolo" /
    "test" /
    "images"
)

TEST_LABEL_DIR = (
    BASE_DIR /
    "datasets" /
    "bdd100k_original_yolo" /
    "test" /
    "labels"
)

CNN_MODEL_PATH = (
    BASE_DIR /
    "results" /
    "final_analysis" /
    "cnn_policy" /
    "models" /
    "cnn_baseline_best.pt"
)

YOLO_MODEL_PATH = (
    BASE_DIR /
    "results" /
    "yolo26_baseline" /
    "bdd100k_original_yolo_yolo26n_baseline_30ep" /
    "weights" /
    "best.pt"
)

OUTPUT_DIR = (
    BASE_DIR /
    "results" /
    "final_analysis" /
    "cnn_policy" /
    "20k_final_test"
)

CNN_IMAGE_DIR = (
    OUTPUT_DIR /
    "cnn_selected" /
    "images"
)

CNN_LABEL_DIR = (
    OUTPUT_DIR /
    "cnn_selected" /
    "labels"
)

IDENTITY_IMAGE_DIR = (
    OUTPUT_DIR /
    "identity" /
    "images"
)

IDENTITY_LABEL_DIR = (
    OUTPUT_DIR /
    "identity" /
    "labels"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CNN_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CNN_LABEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IDENTITY_IMAGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

IDENTITY_LABEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)




DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

IMAGE_SIZE = 224

CLASS_NAMES = [
    "identity",
    "clahe",
    "gamma",
    "dehazing",
    "denoising"
]

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]



class CNNPolicy(nn.Module):

    def __init__(
        self,
        num_classes=5
    ):

        super().__init__()

        backbone = models.resnet18(
            weights=None
        )

        feature_dim = (
            backbone.fc.in_features
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.policy_head = nn.Sequential(
            nn.Linear(
                feature_dim,
                256
            ),
            nn.ReLU(),
            nn.Dropout(
                0.3
            ),
            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(
        self,
        x
    ):

        features = self.backbone(
            x
        )

        return self.policy_head(
            features
        )



cnn_model = CNNPolicy(
    num_classes=5
)

checkpoint = torch.load(
    CNN_MODEL_PATH,
    map_location=DEVICE
)

cnn_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

cnn_model = cnn_model.to(
    DEVICE
)

cnn_model.eval()



cnn_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

def identity(
    image
):

    return image.copy()


def clahe(
    image
):

    lab = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = (
        cv2.split(lab)
    )

    operator = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = operator.apply(
        l_channel
    )

    result = cv2.merge([
        l_channel,
        a_channel,
        b_channel
    ])

    return cv2.cvtColor(
        result,
        cv2.COLOR_LAB2BGR
    )


def gamma_correction(
    image,
    gamma=0.8
):

    table = np.array([
        (
            i / 255.0
        ) ** gamma * 255
        for i in range(256)
    ]).astype(
        np.uint8
    )

    return cv2.LUT(
        image,
        table
    )


def dehazing(
    image,
    omega=0.75,
    patch_size=15,
    t_min=0.20
):

    image_float = (
        image.astype(
            np.float32
        ) / 255.0
    )

    dark = np.min(
        image_float,
        axis=2
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (
            patch_size,
            patch_size
        )
    )

    dark = cv2.erode(
        dark,
        kernel
    )

    atmospheric = np.max(
        image_float.reshape(
            -1,
            3
        ),
        axis=0
    )

    transmission = (
        1.0 -
        omega * dark
    )

    transmission = np.maximum(
        transmission,
        t_min
    )

    result = np.empty_like(
        image_float
    )

    for channel in range(3):

        result[:, :, channel] = (
            (
                image_float[:, :, channel]
                -
                atmospheric[channel]
            )
            /
            transmission
        ) + atmospheric[channel]

    result = np.clip(
        result,
        0.0,
        1.0
    )

    return (
        result * 255
    ).astype(
        np.uint8
    )


def denoising(
    image
):

    tensor = torch.from_numpy(
        image
    ).permute(
        2,
        0,
        1
    ).float()

    tensor = tensor.unsqueeze(
        0
    ).to(
        DEVICE
    )

    filtered = torch.nn.functional.avg_pool2d(
        tensor,
        kernel_size=7,
        stride=1,
        padding=3
    )

    result = (
        0.7 * filtered +
        0.3 * tensor
    )

    return (
        result
        .clamp(
            0,
            255
        )
        .squeeze(0)
        .permute(
            1,
            2,
            0
        )
        .byte()
        .cpu()
        .numpy()
    )


ENHANCEMENT_FUNCTIONS = {
    "identity": identity,
    "clahe": clahe,
    "gamma": gamma_correction,
    "dehazing": dehazing,
    "denoising": denoising
}



image_paths = sorted(
    TEST_IMAGE_DIR.glob(
        "*.jpg"
    )
)

print("=" * 80)
print("CNN 20K FINAL TEST")
print("=" * 80)

print()
print(
    f"Test images found : "
    f"{len(image_paths):,}"
)




policy_results = []

start_time = time.time()

for index, image_path in enumerate(
    image_paths,
    start=1
):

    image_bgr = cv2.imread(
        str(image_path)
    )

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    tensor = cnn_transform(
        Image.fromarray(
            image_rgb
        )
    ).unsqueeze(
        0
    ).to(
        DEVICE,
        non_blocking=True
    )

    with torch.no_grad():

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            logits = cnn_model(
                tensor
            )

    probabilities = torch.softmax(
        logits,
        dim=1
    )

    action_id = int(
        torch.argmax(
            probabilities,
            dim=1
        ).item()
    )

    action_name = (
        CLASS_NAMES[action_id]
    )

    enhanced_image = (
        ENHANCEMENT_FUNCTIONS[
            action_name
        ](
            image_bgr
        )
    )

    cv2.imwrite(
        str(
            CNN_IMAGE_DIR /
            image_path.name
        ),
        enhanced_image
    )

    label_path = (
        TEST_LABEL_DIR /
        f"{image_path.stem}.txt"
    )

    if label_path.exists():

        shutil.copy2(
            label_path,
            CNN_LABEL_DIR /
            label_path.name
        )

    policy_results.append({
        "image": image_path.name,
        "action_id": action_id,
        "action": action_name,
        "confidence": float(
            probabilities[
                0,
                action_id
            ].item()
        )
    })

    if index % 2000 == 0:

        print(
            f"{index:,}/{len(image_paths):,}"
        )


policy_df = pd.DataFrame(
    policy_results
)

policy_path = (
    OUTPUT_DIR /
    "cnn_policy_20k_predictions.csv"
)

policy_df.to_csv(
    policy_path,
    index=False
)


for image_path in image_paths:

    shutil.copy2(
        image_path,
        IDENTITY_IMAGE_DIR /
        image_path.name
    )

    label_path = (
        TEST_LABEL_DIR /
        f"{image_path.stem}.txt"
    )

    if label_path.exists():

        shutil.copy2(
            label_path,
            IDENTITY_LABEL_DIR /
            label_path.name
        )



CLASS_DEFINITIONS = """
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: train
  6: truck
  7: traffic light
  8: fire hydrant
  9: stop sign
  10: parking meter
  11: bench
  12: bird
  13: cat
  14: dog
  15: horse
  16: sheep
  17: cow
  18: elephant
  19: bear
  20: zebra
  21: giraffe
  22: backpack
  23: umbrella
  24: handbag
"""

CNN_YAML = (
    OUTPUT_DIR /
    "cnn_selected_20k.yaml"
)

IDENTITY_YAML = (
    OUTPUT_DIR /
    "identity_20k.yaml"
)

CNN_YAML.write_text(
    (
        f"path: {OUTPUT_DIR}\n"
        "train: cnn_selected/images\n"
        "val: cnn_selected/images\n"
        f"{CLASS_DEFINITIONS}"
    ),
    encoding="utf-8"
)

IDENTITY_YAML.write_text(
    (
        f"path: {OUTPUT_DIR}\n"
        "train: identity/images\n"
        "val: identity/images\n"
        f"{CLASS_DEFINITIONS}"
    ),
    encoding="utf-8"
)



yolo_model = YOLO(
    str(
        YOLO_MODEL_PATH
    )
)


identity_metrics = yolo_model.val(
    data=str(
        IDENTITY_YAML
    ),
    imgsz=640,
    conf=0.25,
    iou=0.50,
    device=0,
    verbose=False
)



cnn_metrics = yolo_model.val(
    data=str(
        CNN_YAML
    ),
    imgsz=640,
    conf=0.25,
    iou=0.50,
    device=0,
    verbose=False
)


def calculate_f1(
    precision,
    recall
):

    return (
        2.0 *
        precision *
        recall /
        (
            precision +
            recall +
            1e-12
        )
    )


results_df = pd.DataFrame([
    {
        "method": "Identity",
        "precision": float(
            identity_metrics.box.mp
        ),
        "recall": float(
            identity_metrics.box.mr
        ),
        "F1": calculate_f1(
            identity_metrics.box.mp,
            identity_metrics.box.mr
        ),
        "mAP50": float(
            identity_metrics.box.map50
        ),
        "mAP50_95": float(
            identity_metrics.box.map
        )
    },
    {
        "method": "CNN-selected",
        "precision": float(
            cnn_metrics.box.mp
        ),
        "recall": float(
            cnn_metrics.box.mr
        ),
        "F1": calculate_f1(
            cnn_metrics.box.mp,
            cnn_metrics.box.mr
        ),
        "mAP50": float(
            cnn_metrics.box.map50
        ),
        "mAP50_95": float(
            cnn_metrics.box.map
        )
    }
])

results_path = (
    OUTPUT_DIR /
    "cnn_final_20k_test_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)



print()
print("=" * 80)
print("CNN 20K FINAL TEST COMPLETED")
print("=" * 80)

print()
print(
    results_df.to_string(
        index=False
    )
)

print()
print(
    f"Policy predictions:"
)

print(
    policy_path
)

print()
print(
    f"Final results:"
)

print(
    results_path
)

print()
print("✓ CNN checkpoint used.")
print("✓ 20K test set evaluated.")
print("✓ CNN-selected enhancement evaluated.")
print("✓ Identity baseline evaluated.")
print("✓ Final CNN test results saved.")

CNN 20K FINAL TEST

Test images found : 20,000
2,000/20,000
4,000/20,000
6,000/20,000
8,000/20,000
10,000/20,000
12,000/20,000
14,000/20,000
16,000/20,000
18,000/20,000
20,000/20,000
Ultralytics 8.4.18  Python-3.10.11 torch-2.12.0.dev20260226+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 13.15.4 MB/s, size: 59.5 KB)
val: Scanning E:\ML_Project\results\final_analysis\cnn_policy\20k_final_test\identity\labels... 20000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20000/20000 771.8it/s 25.9s<0.1s
val: E:\ML_Project\results\final_analysis\cnn_policy\20k_final_test\identity\images\e6f10c58-c46de527.jpg: 1 duplicate labels removed
val: New cache created: E:\ML_Project\results\final_analysis\cnn_policy\20k_final_test\identity\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1250/12